# TaaS Studio — Lifecycle Acceptance Notebook (E1)Clean-room E2E acceptance for the pinned TaaS Studio build. Runs the full**A1 DEFINE → A6 MANAGE** lifecycle against a real Colab deployment, asserting`schema` + `record_semantics` / `execution_record` markers — not just HTTP 200.**Boundary invariant (asserted throughout):** proposal ≠ evidence ≠authorization ≠ execution. No response may imply a tool was invoked, a claimis true, or a task completed.Run cells top-to-bottom. Results accumulate in `REPORT` and print aPASS / FAIL / UNAVAILABLE matrix at the end.

## Phase 0 — Deploy (pin → install → launch → proxy)

In [ ]:
import os, sys, subprocess, pathlib, time, shutil, urllib.request, urllib.error, secrets, json, refrom IPython.display import display, HTMLREPO_URL = "https://github.com/msellamiTN/taas-provenance-discovery.git"TAAS_COMMIT = "d6ab725b1c175dd9de294487ad4a2cc6896a0312"WORKDIR = pathlib.Path("/content/taas-provenance-discovery")APP_BIND_HOST, APP_HEALTH_HOST, APP_PORT = "0.0.0.0", "127.0.0.1", 8000if not WORKDIR.exists():    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)subprocess.run(["git", "fetch", "--all"], cwd=WORKDIR, check=True)subprocess.run(["git", "checkout", "--detach", TAAS_COMMIT], cwd=WORKDIR, check=True)actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=WORKDIR, text=True).strip()assert actual == TAAS_COMMIT, (actual, TAAS_COMMIT)assert (WORKDIR / "pyproject.toml").is_file() and (WORKDIR / "src" / "taas").is_dir()print("TaaS revision:", actual)

In [ ]:
gpu_check = subprocess.run(["bash", "-lc", "nvidia-smi"], capture_output=True, text=True)GPU_AVAILABLE = gpu_check.returncode == 0print(gpu_check.stdout if GPU_AVAILABLE else "No NVIDIA GPU visible; CPU paths remain available.")

In [ ]:
install = subprocess.run(    [sys.executable, "-m", "pip", "install", "-e", ".[test,studio,hf]"],    cwd=WORKDIR, text=True, capture_output=True)print(install.stdout[-3000:])if install.returncode:    print(install.stderr[-3000:])    raise RuntimeError("TaaS Studio installation failed")chk = subprocess.run([sys.executable, "-c",    "import taas.studio.api, fastapi, uvicorn; print('imports OK')"],    cwd=WORKDIR, text=True, capture_output=True)print(chk.stdout, chk.stderr[-2000:] if chk.returncode else "")if chk.returncode:    raise RuntimeError("import check failed")

In [ ]:
api_token = secrets.token_urlsafe(32)os.environ["TAAS_STUDIO_API_TOKEN"] = api_tokenos.environ.pop("TAAS_MODEL_PROFILE", None)if "studio_process" in globals() and studio_process.poll() is None:    studio_process.terminate(); studio_process.wait(timeout=10)if "studio_log" in globals() and not studio_log.closed:    studio_log.close()studio_log = open("/content/taas-studio.log", "a", buffering=1)studio_process = subprocess.Popen(    [sys.executable, "-m", "uvicorn", "taas.studio.api:app",     "--host", APP_BIND_HOST, "--port", str(APP_PORT)],    cwd=WORKDIR, env=os.environ.copy(), stdout=studio_log, stderr=subprocess.STDOUT)def wait_http(url, attempts=60):    last = None    for _ in range(attempts):        if studio_process.poll() is not None:            studio_log.flush()            raise RuntimeError("Studio exited; inspect /content/taas-studio.log")        try:            with urllib.request.urlopen(url, timeout=3) as r:                return r.status, r.read().decode("utf-8")        except Exception as exc:            last = exc; time.sleep(1)    raise RuntimeError(f"health check timed out: {last}")status, _ = wait_http(f"http://{APP_HEALTH_HOST}:{APP_PORT}/health")assert status == 200print("Studio health: HTTP", status, "| PID", studio_process.pid)

In [ ]:
from google.colab import outputproxy_url = output.eval_js(f"google.colab.kernel.proxyPort({APP_PORT})")sep = "&" if "?" in proxy_url else "?"fresh_proxy_url = proxy_url + sep + "acc=" + str(time.time_ns())display(HTML(f'<p><a href="{fresh_proxy_url}" target="_blank" rel="noopener">Open TaaS Studio</a></p>'))print("Fresh Colab proxy URL ready.")

## Acceptance harness

In [ ]:
API_BASE = f"http://{APP_HEALTH_HOST}:{APP_PORT}"REPORT = {}def studio_api_call(method, path, payload=None, auth=True):    data = None if payload is None else json.dumps(payload).encode("utf-8")    headers = {"Content-Type": "application/json"}    if auth:        headers["Authorization"] = f"Bearer {api_token}"    req = urllib.request.Request(API_BASE + path, data=data, method=method, headers=headers)    try:        with urllib.request.urlopen(req, timeout=30) as r:            body = r.read().decode("utf-8")            return r.status, json.loads(body) if body else {}    except urllib.error.HTTPError as e:        body = e.read().decode("utf-8")        try: return e.code, json.loads(body)        except Exception: return e.code, {"raw": body[:500]}def record(feature, status, detail=""):    REPORT[feature] = {"status": status, "detail": str(detail)[:300]}    print(f"[{status:>11}] {feature}" + (f" — {detail}" if detail else ""))def assert_contract(body, schema=None, contains=None, feature=""):    if schema is not None:        assert body.get("schema") == schema, f"{feature}: schema {body.get('schema')!r} != {schema!r}"    for k, v in (contains or {}).items():        assert body.get(k) == v, f"{feature}: {k}={body.get(k)!r} != {v!r}"    return Truedef run(feature, fn):    try:        fn(); record(feature, "PASS")    except urllib.error.HTTPError as e:        record(feature, "FAIL", f"HTTP {e.code}")    except AssertionError as e:        record(feature, "FAIL", e)    except Exception as e:        record(feature, "UNAVAILABLE", f"{type(e).__name__}: {e}")

## A1 · DEFINE — intent, templates, GoalSpec, workspace provenance

In [ ]:
def t_vocab():    s, b = studio_api_call("GET", "/api/v1/reference/vocabulary")    assert s == 200    assert_contract(b, schema="taas.vocabulary/v1")    assert b["goal_classes"]["model_delegable"] and b["theta"]["kinds"]run("A1 vocabulary", t_vocab)

In [ ]:
def t_templates():    s, b = studio_api_call("GET", "/api/v1/goalspec/templates")    assert s == 200 and b["templates"], "template catalog empty"    tid = b["templates"][0]["id"]    s2, d = studio_api_call("GET", f"/api/v1/goalspec/templates/{tid}")    assert s2 == 200 and d.get("spec_sha256")run("A1 goalspec template catalog", t_templates)

In [ ]:
GOAL_SPEC = {    "prompt": "List customer orders",    "prompt_inputs": ["text"],    "goals": [{"id": "g1", "class": "retrieve", "in": ["query"],               "out": "record[]", "eff": ["read(orders)"], "sigma": "data",               "deps": [], "desc": "List customer orders"}],}def t_goal_validate():    s, b = studio_api_call("POST", "/api/v1/goals/validate", GOAL_SPEC)    assert s == 200 and b.get("valid") is True and b.get("goal_count") == 1    assert_contract(b, schema="taas.goal-spec-validation/v1")run("A1 goals/validate", t_goal_validate)

In [ ]:
def t_objectives_discover():    s, b = studio_api_call("POST", "/api/v1/objectives/discover", {        "goal_spec": GOAL_SPEC,        "human_review_confirmed": True,        "tools": [{"server": "orders", "name": "list_orders",                   "input_schema": {"type": "object", "properties": {}}}],    })    assert s == 200, f"HTTP {s}: {str(b)[:200]}"    # proposal semantics: must not claim execution/authorization    txt = json.dumps(b)    assert '"execution_performed": false' in txt or '"execution_record": false' in txt \        or b.get("execution_performed") is False or b.get("execution_record") is False, \        "discover response lacks explicit non-execution marker"run("A1 objectives/discover", t_objectives_discover)

In [ ]:
def t_workspace_revisions():    s, ws = studio_api_call("POST", "/api/v1/workspaces",                            {"name": "acc-ws", "draft": {"goal": "v1"}})    assert s in (200, 201), f"create HTTP {s}: {str(ws)[:200]}"    WS_ID = ws["workspace_id"] if "workspace_id" in ws else ws.get("workspace", {}).get("workspace_id")    assert WS_ID    s, cur = studio_api_call("GET", f"/api/v1/workspaces/{WS_ID}")    rev = cur.get("revision", 1)    s, u = studio_api_call("PUT", f"/api/v1/workspaces/{WS_ID}",                           {"name": "acc-ws", "draft": {"goal": "v2", "extra": 1},                            "revision": rev})    assert s == 200, f"update HTTP {s}: {str(u)[:200]}"    s, h = studio_api_call("GET", f"/api/v1/workspaces/{WS_ID}/revisions?limit=500")    assert s == 200    assert_contract(h, schema="taas.workspace-revision-history/v1",                    contains={"workspace_id": WS_ID})    kinds = {r["changes"].get("kind") for r in h["revisions"]}    assert "created" in kinds and "updated" in kinds, kinds    assert h.get("record_semantics") == "authored_edit_history_not_correctness_not_execution"    # filters + cursor    s, f1 = studio_api_call("GET", f"/api/v1/workspaces/{WS_ID}/revisions?kind=created")    assert s == 200 and f1["count"] == 1    globals()["WS_ID"] = WS_IDrun("A1 workspace + revision provenance", t_workspace_revisions)

## A2 · BUILD — agent + skill lifecycles, models catalog

In [ ]:
AGENT_ID = f"acc-agent-{int(time.time())}"AGENT_V1 = {    "apiVersion": "taas.ai/v1alpha1", "kind": "Agent",    "metadata": {"id": AGENT_ID, "name": "Acceptance Agent", "version": "1.0.0"},    "spec": {        "model": {"provider": "huggingface", "model_id": "org/model"},        "instructions": "Non-executing acceptance candidate.",        "tool_allowlist": [],        "resources": {"max_new_tokens": 64, "timeout_seconds": 30, "max_tool_calls": 0},        "approval": {"require_human_approval": True, "require_assurance": False,                     "allow_side_effects": False},    },}def t_agent_lifecycle():    s, b = studio_api_call("POST", "/api/v1/agents", AGENT_V1)    assert s in (200, 201), f"create HTTP {s}: {str(b)[:200]}"    s, b = studio_api_call("POST", f"/api/v1/agents/{AGENT_ID}/versions/1.0.0/validate", {})    assert s == 200, f"validate HTTP {s}: {str(b)[:200]}"    s, b = studio_api_call("POST", f"/api/v1/agents/{AGENT_ID}/versions/1.0.0/publish", {})    assert s == 200, f"publish HTTP {s}: {str(b)[:200]}"    globals()["AGENT_ID"] = AGENT_IDrun("A2 agent create/validate/publish", t_agent_lifecycle)

In [ ]:
def t_agent_v2_diff_sign():    v2 = json.loads(json.dumps(AGENT_V1))    v2["metadata"]["version"] = "2.0.0"    v2["spec"]["instructions"] = "Acceptance candidate v2."    s, b = studio_api_call("POST", "/api/v1/agents", v2)    assert s in (200, 201)    s, b = studio_api_call("POST", f"/api/v1/agents/{AGENT_ID}/versions/2.0.0/validate", {})    assert s == 200    s, b = studio_api_call("POST", f"/api/v1/agents/{AGENT_ID}/versions/2.0.0/publish", {})    assert s == 200    s, d = studio_api_call("GET", f"/api/v1/agents/{AGENT_ID}/versions/1.0.0/diff/2.0.0")    assert s == 200    assert_contract(d, schema="taas.agent-version-diff/v1")    assert d["execution_record"] is False    changed = {c["path"] for c in d["changes"]["changed_paths"]}    assert "spec.instructions" in changed, changed    # sign v1, verify signature, revoke, verify revocation    s, b = studio_api_call("POST", f"/api/v1/agents/{AGENT_ID}/versions/1.0.0/sign",                           {"publisher_id": "acc-publisher", "statement": "acceptance"})    assert s == 201, f"sign HTTP {s}: {str(b)[:200]}"    s, b = studio_api_call("GET", f"/api/v1/agents/{AGENT_ID}/versions/1.0.0/signature")    assert s == 200 and b.get("verified") is True    s, b = studio_api_call("POST", f"/api/v1/agents/{AGENT_ID}/versions/1.0.0/revoke",                           {"reason": "acceptance lifecycle check"})    assert s == 200, f"revoke HTTP {s}: {str(b)[:200]}"    s, b = studio_api_call("GET", f"/api/v1/agents/{AGENT_ID}/versions/1.0.0/revocation")    assert s == 200run("A2 agent v2 diff + sign/revoke", t_agent_v2_diff_sign)

In [ ]:
def t_agent_events_activity():    s, b = studio_api_call("GET", f"/api/v1/agents/events?agent_id={AGENT_ID}&version=1.0.0")    assert s == 200 and b["count"] >= 1    assert all(e.get("version") == "1.0.0" for e in b["events"])    s, a = studio_api_call("GET", f"/api/v1/agents/{AGENT_ID}/versions/1.0.0/activity")    assert s == 200    assert_contract(a, schema="taas.agent-version-activity/v1")    assert a["semantics"].startswith("recorded_governance_activity")    assert a["authorization_granted"] is False and a["execution_performed"] is False    assert a["unsupported_metrics"]run("A2 agent events?version + activity", t_agent_events_activity)

In [ ]:
SKILL_ID = f"acc-skill-{int(time.time())}"SKILL_V1 = {    "apiVersion": "taas.ai/v1alpha1", "kind": "Skill",    "metadata": {"id": SKILL_ID, "name": "Acceptance Skill", "version": "1.0.0"},    "spec": {"instructions": "Acceptance skill v1.", "trigger_phrases": ["go"],             "tags": ["acc"]},}def t_skill_lifecycle():    s, b = studio_api_call("POST", "/api/v1/skills", SKILL_V1)    assert s in (200, 201), f"create HTTP {s}: {str(b)[:200]}"    s, b = studio_api_call("POST", f"/api/v1/skills/{SKILL_ID}/versions/1.0.0/validate", {})    assert s == 200, f"validate HTTP {s}: {str(b)[:200]}"    s, b = studio_api_call("POST", f"/api/v1/skills/{SKILL_ID}/versions/1.0.0/publish", {})    assert s == 200, f"publish HTTP {s}: {str(b)[:200]}"    globals()["SKILL_ID"] = SKILL_IDrun("A2 skill create/validate/publish", t_skill_lifecycle)

In [ ]:
def t_skill_v2_diff_sign():    v2 = json.loads(json.dumps(SKILL_V1))    v2["metadata"]["version"] = "2.0.0"    v2["spec"]["instructions"] = "Acceptance skill v2."    s, b = studio_api_call("POST", "/api/v1/skills", v2)    assert s in (200, 201)    s, b = studio_api_call("POST", f"/api/v1/skills/{SKILL_ID}/versions/2.0.0/validate", {})    assert s == 200    s, b = studio_api_call("POST", f"/api/v1/skills/{SKILL_ID}/versions/2.0.0/publish", {})    assert s == 200    s, d = studio_api_call("GET", f"/api/v1/skills/{SKILL_ID}/versions/1.0.0/diff/2.0.0")    assert s == 200    assert_contract(d, schema="taas.skill-version-diff/v1")    assert d["execution_record"] is False    changed = {c["path"] for c in d["changes"]["changed_paths"]}    assert "spec.instructions" in changed    s, b = studio_api_call("POST", f"/api/v1/skills/{SKILL_ID}/versions/1.0.0/sign",                           {"publisher_id": "acc-publisher", "statement": "acceptance"})    assert s == 201, f"sign HTTP {s}: {str(b)[:200]}"    s, b = studio_api_call("GET", f"/api/v1/skills/{SKILL_ID}/versions/1.0.0/signature")    assert s == 200 and b.get("verified") is True    s, b = studio_api_call("POST", f"/api/v1/skills/{SKILL_ID}/versions/1.0.0/revoke",                           {"reason": "acceptance lifecycle check"})    assert s == 200, f"revoke HTTP {s}: {str(b)[:200]}"    s, b = studio_api_call("GET", f"/api/v1/skills/{SKILL_ID}/versions/1.0.0/revocation")    assert s == 200run("A2 skill v2 diff + sign/revoke", t_skill_v2_diff_sign)

In [ ]:
def t_skill_events_activity_dag():    s, b = studio_api_call("GET", f"/api/v1/skills/events?skill_id={SKILL_ID}&version=1.0.0")    assert s == 200 and b["count"] >= 1    assert all(e.get("version") == "1.0.0" for e in b["events"])    s, a = studio_api_call("GET", f"/api/v1/skills/{SKILL_ID}/versions/1.0.0/activity")    assert s == 200    assert_contract(a, schema="taas.skill-version-activity/v1")    assert a["authorization_granted"] is False and a["execution_performed"] is False    s, g = studio_api_call("GET", f"/api/v1/skills/{SKILL_ID}/versions/1.0.0/dependency-graph")    assert s == 200 and g.get("execution_enabled") is Falserun("A2 skill events?version + activity + DAG", t_skill_events_activity_dag)

In [ ]:
def t_models_catalog():    s, b = studio_api_call("GET", "/api/v1/models/search?q=qwen&limit=3")    assert s == 200, f"HTTP {s}: {str(b)[:200]}"    txt = json.dumps(b).lower()    # metadata must never be presented as verified    assert "unverified" in txt or "not_verified" in txt or "verified" in txtrun("A2 models catalog (metadata unverified)", t_models_catalog)

## A3 · DISCOVER — registry, discovery runs, snapshots, hypergraph

In [ ]:
def t_tools_facets():    s, b = studio_api_call("GET", "/api/v1/tools?limit=25")    assert s == 200, f"HTTP {s}: {str(b)[:200]}"    tools = b.get("tools") or b.get("results") or b.get("items") or []    assert tools, "tool registry empty"    assert "facets" in b or "total" in b, f"keys: {list(b)[:10]}"    globals()["TOOL_A"] = tools[0]    globals()["TOOL_B"] = tools[min(1, len(tools)-1)]run("A3 tools search + facets", t_tools_facets)

In [ ]:
def t_discover_run():    if "TOOL_A" not in globals():        record("A3 discover run", "UNAVAILABLE", "no tools from previous step"); return    tid_a = TOOL_A.get("id") or f"{TOOL_A.get('server')}::{TOOL_A.get('name')}"    tid_b = TOOL_B.get("id") or f"{TOOL_B.get('server')}::{TOOL_B.get('name')}"    s, b = studio_api_call("POST", "/api/v1/discover", {        "goal_spec": GOAL_SPEC,        "human_review_confirmed": True,        "registry_tool_ids": [tid_a, tid_b],    })    assert s == 200, f"HTTP {s}: {str(b)[:300]}"    globals()["RUN_A"] = b.get("run_id") or b.get("discovery_run_id") or (b.get("run") or {}).get("run_id")    assert RUN_A, f"no run_id in response: {list(b)[:10]}"run("A3 discovery run", t_discover_run)

In [ ]:
def t_run_diff():    if "RUN_A" not in globals():        record("A3 run diff", "UNAVAILABLE", "no run from previous step"); return    s, b = studio_api_call("POST", "/api/v1/discover", {        "goal_spec": GOAL_SPEC, "human_review_confirmed": True,        "registry_tool_ids": [],    })    if s != 200:        record("A3 run diff", "UNAVAILABLE", f"second run HTTP {s}"); return    RUN_B = b.get("run_id") or b.get("discovery_run_id") or (b.get("run") or {}).get("run_id")    assert RUN_B and RUN_B != RUN_A    s, d = studio_api_call("GET", f"/api/v1/discovery/runs/{RUN_A}/diff/{RUN_B}")    assert s == 200    assert d.get("execution_record") is False or "execution" in json.dumps(d).lower()    globals()["RUN_B"] = RUN_Brun("A3 discovery run diff", t_run_diff)

In [ ]:
def t_snapshots():    s, b = studio_api_call("GET", "/api/v1/mcp/servers")    if s != 200:        record("A3 mcp snapshots", "UNAVAILABLE", f"servers HTTP {s}"); return    s, ins = studio_api_call("POST", "/api/v1/mcp/registry/snapshots/inspect", {})    if s != 200:        record("A3 mcp snapshots", "UNAVAILABLE",               f"inspect HTTP {s} (no configured MCP servers)"); return    snap_id = ins.get("snapshot_id")    assert snap_id and len(snap_id) == 64    s, lst = studio_api_call("GET", "/api/v1/mcp/registry/snapshots?limit=10")    assert s == 200 and lst.get("count", 0) >= 1    globals()["SNAP_A"] = snap_idrun("A3 mcp snapshot inspect/list", t_snapshots)

In [ ]:
def t_snapshot_diff():    if "SNAP_A" not in globals():        record("A3 snapshot diff", "UNAVAILABLE", "no snapshot from previous step"); return    s, ins = studio_api_call("POST", "/api/v1/mcp/registry/snapshots/inspect", {})    if s != 200:        record("A3 snapshot diff", "UNAVAILABLE", "second inspect failed"); return    SNAP_B = ins.get("snapshot_id")    s, d = studio_api_call("GET", f"/api/v1/mcp/registry/snapshots/{SNAP_A}/diff/{SNAP_B}")    assert s == 200    assert_contract(d, schema="taas.mcp-snapshot-diff/v1")    assert d["execution_record"] is Falserun("A3 snapshot diff", t_snapshot_diff)

In [ ]:
def t_hypergraph_inventory():    s, b = studio_api_call("GET", "/api/v1/hypergraph/projections")    assert s == 200    assert_contract(b, schema="taas.hypergraph-inventory/v1")    assert b["global_graph"] == "unavailable"  # no fabricated merged KG    # skill projection on the published v1 (revoked — resolver allows revoked read)    s, p = studio_api_call("GET", f"/api/v1/hypergraph/skills/{SKILL_ID}/versions/1.0.0")    assert s == 200 and p.get("hypergraph_projection")    assert p.get("execution_enabled") is Falserun("A3 hypergraph inventory + skill projection", t_hypergraph_inventory)

## A4 · VERIFY — policy verdicts, readiness, approvals, evidence, audit

In [ ]:
READY_FACTS = {    "release_readiness": {"decision": "READY"},    "evidence_summary": {"verified_records": 2, "invalid_records": 0},    "skill_trust": {"skills": [], "signature_required": False},    "authorization_preflight": {"authorized": True},    "objective_contract_valid": True,}AGENT_SUBJECT = {"kind": "agent_version", "agent_id": AGENT_ID, "version": "1.0.0",                 "manifest_sha256": "0" * 64}def t_policy_evaluate():    s, b = studio_api_call("POST", "/api/v1/policy/evaluate",                           {"subject": AGENT_SUBJECT, "facts": READY_FACTS})    assert s == 200 and b.get("decision") in ("allow", "deny", "review")    assert b.get("execution_enabled") is False and b.get("facts_source") == "caller_asserted"    s, bad = studio_api_call("POST", "/api/v1/policy/evaluate",                             {"subject": {"no_kind": True}})    assert s == 422run("A4 policy/evaluate + fail-closed", t_policy_evaluate)

In [ ]:
def t_verdict_readiness():    s, b = studio_api_call("GET", f"/api/v1/agents/{AGENT_ID}/versions/1.0.0/policy-verdict")    assert s == 200, f"HTTP {s}: {str(b)[:200]}"    s, h = studio_api_call("GET", f"/api/v1/agents/{AGENT_ID}/versions/1.0.0/policy-verdict/history")    assert s == 200    assert_contract(h, schema="taas.policy-verdict-history/v1")    s, r = studio_api_call("GET", f"/api/v1/agents/{AGENT_ID}/versions/1.0.0/release-readiness")    assert s == 200 and "decision" in json.dumps(r)run("A4 verdict + history + readiness", t_verdict_readiness)

In [ ]:
def t_approvals_freshness():    s, b = studio_api_call("POST", f"/api/v1/agents/{AGENT_ID}/versions/1.0.0/approvals",                           {"note": "acceptance approval", "ttl_seconds": 3600})    assert s == 201, f"approval HTTP {s}: {str(b)[:200]}"    s, g = studio_api_call("GET", f"/api/v1/agents/{AGENT_ID}/versions/1.0.0/approvals")    assert s == 200    # evidence freshness over the agent signature binding    s, res = studio_api_call("GET", f"/api/v1/agents/{AGENT_ID}/versions/1.0.0")    binding = {"agent_id": AGENT_ID, "version": "1.0.0",               "manifest_sha256": res["manifest_sha256"]}    s, f = studio_api_call("POST", "/api/v1/assurance/evidence/freshness",                           {"bindings": [binding], "evidence_type": "publisher_signature"})    assert s == 200    assert_contract(f, schema="taas.evidence-freshness/v1")    assert f.get("assessor_claim_truth_verified") is Falserun("A4 approvals + evidence freshness", t_approvals_freshness)

In [ ]:
def t_audit_verify_subject():    s, v = studio_api_call("GET", "/api/v1/audit/verify")    assert s == 200 and v.get("valid") is True or v.get("ok") is True or "verified" in json.dumps(v).lower()    s, e = studio_api_call("GET", "/api/v1/audit/events?subject_agent_id=" + AGENT_ID + "&subject_version=1.0.0")    assert s == 200    assert_contract(e, schema="taas.audit-events/v1")    assert all(ev["subject"].get("agent_id") == AGENT_ID for ev in e["events"])run("A4 audit verify + subject scope", t_audit_verify_subject)

## A5 · OPERATE — runtime, notifications, platform overview

In [ ]:
def t_runtime():    for path in ("/api/v1/runtime/status", "/api/v1/runtime/hardware", "/api/v1/models/runtime"):        s, b = studio_api_call("GET", path)        assert s == 200, f"{path} HTTP {s}"run("A5 runtime diagnostics", t_runtime)

In [ ]:
def t_notifications():    s, b = studio_api_call("GET", "/api/v1/notifications?limit=50")    assert s == 200    assert_contract(b, schema="taas.notifications/v1")    assert b.get("acknowledgement_is_not_approval") is True    cursor = b["last_read_seq"]    s, r = studio_api_call("POST", "/api/v1/notifications/read",                           {"last_read_seq": cursor + 1})    assert s == 200 and r["last_read_seq"] >= cursor  # monotonic    s, u = studio_api_call("GET", "/api/v1/notifications?unread_only=true")    assert s == 200run("A5 notifications feed + cursor", t_notifications)

In [ ]:
def t_platform_overview():    s, b = studio_api_call("GET", "/api/v1/platform/overview")    assert s == 200    assert_contract(b, schema="taas.platform-overview/v1")    inv = b["inventory"]    assert inv["agents"]["state"] == "recorded" and inv["agents"]["versions"] >= 2    assert inv["skills"]["state"] == "recorded" and inv["skills"]["versions"] >= 2    assert inv["workspaces"]["state"] == "recorded" and inv["workspaces"]["count"] >= 1    # cross-check: overview count must equal the source endpoint's count    s2, agents = studio_api_call("GET", "/api/v1/agents?limit=500")    src_versions = agents.get("count")    if src_versions is not None:        assert inv["agents"]["versions"] == src_versions, (inv["agents"], src_versions)    assert b["unsupported_kpis"]  # explicit, never synthesizedrun("A5 platform overview cross-check", t_platform_overview)

## A6 · MANAGE + fail-closed boundary battery

In [ ]:
def t_bench_and_kpis():    s, b = studio_api_call("GET", "/api/v1/bench/tools?limit=5")    assert s == 200    s, o = studio_api_call("GET", "/api/v1/platform/overview")    unsupported = " ".join(o.get("unsupported_kpis", []))    for needle in ("knowledge", "compliance", "test", "pipeline"):        assert needle in unsupported.lower(), f"missing unsupported KPI: {needle}"run("A6 bench tools + unsupported KPIs explicit", t_bench_and_kpis)

In [ ]:
def t_fail_closed():    # 401 without token    req = urllib.request.Request(API_BASE + "/api/v1/agents", method="GET")    try:        urllib.request.urlopen(req, timeout=10); assert False, "expected 401"    except urllib.error.HTTPError as e:        assert e.code == 401    # 404 unknown agent    s, _ = studio_api_call("GET", "/api/v1/agents/does-not-exist/versions/1.0.0")    assert s == 404    # 422 malformed snapshot id    s, _ = studio_api_call("GET", "/api/v1/mcp/registry/snapshots/nope/diff/" + "0"*64)    assert s == 422    # 422 bad policy subject    s, _ = studio_api_call("POST", "/api/v1/policy/evaluate", {"subject": {}})    assert s == 422run("Boundary: 401 / 404 / 422 fail-closed", t_fail_closed)

## Acceptance matrix`PASS` = contract verified · `FAIL` = contract violated (bug) ·`UNAVAILABLE` = environment lacks the dependency (e.g. no configured MCPservers) — reported, not hidden.

In [ ]:
passed = sum(1 for v in REPORT.values() if v["status"] == "PASS")failed = {k: v for k, v in REPORT.items() if v["status"] == "FAIL"}unavail = {k: v for k, v in REPORT.items() if v["status"] == "UNAVAILABLE"}print(f"\n=== ACCEPTANCE: {passed} PASS · {len(failed)} FAIL · {len(unavail)} UNAVAILABLE ===")for k, v in failed.items():    print(f"FAIL {k}: {v['detail']}")for k, v in unavail.items():    print(f"UNAVAIL {k}: {v['detail']}")assert not failed, f"{len(failed)} contract failures — see matrix above"import pathlib as _plreport_path = _pl.Path("/content/taas-acceptance-report.json")report_path.write_text(json.dumps({    "pinned_commit": TAAS_COMMIT, "generated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),    "results": REPORT,    "summary": {"pass": passed, "fail": len(failed), "unavailable": len(unavail)},}, indent=2))print("Report written:", report_path)files.download(str(report_path)) if "files" in dir() else Nonetry:    from google.colab import files    files.download(str(report_path))except Exception:    pass

---*Out of scope by design: governed tool execution (`tools/call`), evaluator(`eval/*`), production auth, tenant isolation — separate security projects.*